# SAMHA Single-Image Test Inference

This notebook runs **one test image** through the full SAMHA inference pipeline in small, manageable steps.

**Steps:**
1. Import libraries & setup paths
2. Configure dataset and model parameters
3. Setup GPU and transforms
4. Define helper functions
5. Load test image from dataset
6. Resolve and validate checkpoint
7. Load model
8. Run inference
9. Visualize results

Edit `DATA_ROOT` and `CHECKPOINT_PATH` in **Cell 3**, then run cells top-to-bottom.

In [ ]:
import sys
import os
from pathlib import Path

os.environ['CUDA_VISIBLE_DEVICES'] = '0,1'

import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms

REPO_ROOT = Path('/data_hdd1/users/IBAD/Research/SAMHA')
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from dataset.dataloader import DATALOADER, is_image_file
from utils.trainer_utils import (
    create_model_load_weights,
    global_to_patch,
    global_to_context_patches,
    stitch_patch_predictions_to_global,
)

print('✓ Imports successful')
print('✓ Repo root:', REPO_ROOT)
print('✓ Working dir:', os.getcwd())
print('✓ CUDA_VISIBLE_DEVICES:', os.environ.get('CUDA_VISIBLE_DEVICES'))
print('✓ GPU count seen by PyTorch:', torch.cuda.device_count())

In [ ]:
from typing import Optional

DATA_ROOT = Path('/data_hdd1/users/IBAD/Research/FCtL/data_1/ovarian')
DATASET_ID = 1  # 1: Dataset1, 2: Dataset2
SPLIT = 'test'
IMAGE_NAME: Optional[str] = 'img_06.png' # Set to None to run on all images in the split

# Model parameters
N_CLASS = 2
INPUT_MODE = 3  # (1: local), (2: local + medium), (3: local + medium + large)
USE_WINDOW = True

# Patch & inference parameters
PATCH_SIZE = (512, 512)
OVERLAP = 0.20
SUB_BATCH_SIZE = 2
CONTEXT_M = 2  # medium context multiplier
CONTEXT_L = 3  # large context multiplier

CHECKPOINT_PATH = None  # Set to your checkpoint path (absolute or relative)
REQUIRE_CHECKPOINT = False  # Set to True to enforce checkpoint loading

print('✓ Configuration loaded')
print(f'  Dataset ID: {DATASET_ID}')
print(f'  Input mode: {INPUT_MODE}')
print(f'  Patch size: {PATCH_SIZE}')
print(f'  Checkpoint required: {REQUIRE_CHECKPOINT}')

In [ ]:
# GPU & transform setup
if not torch.cuda.is_available():
    raise RuntimeError('CUDA GPU required for this demo.')

DEVICE = torch.device('cuda')
imagenet_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

print(f'✓ Device: {DEVICE}')
print(f'  GPU: {torch.cuda.get_device_name(0)}')
print(f'  CUDA capability: compute_capability={torch.cuda.get_device_capability(0)}')

In [ ]:
# Helper 1: Find image by name in split directory
def find_image_name(split_root: Path, image_name: Optional[str] = None) -> str:
    image_dir = split_root / 'images'
    if not image_dir.exists():
        raise FileNotFoundError(f'Missing: {image_dir}')

    candidates = [p.name for p in sorted(image_dir.iterdir()) 
                  if p.is_file() and is_image_file(p.name)]
    if not candidates:
        raise FileNotFoundError(f'No images in: {image_dir}')

    if image_name is None:
        return candidates[0]

    if image_name not in candidates:
        raise FileNotFoundError(
            f'{image_name!r} not found. Available: {candidates[0]!r}'
        )
    return image_name

print('✓ find_image_name() defined')

In [ ]:
# Helper 2: Load single image from dataset and detect label folder
def find_label_dir(split_root: Path):
    for name in ('gt', 'masks', 'mask'):
        candidate = split_root / name
        if candidate.exists():
            return candidate
    return None


def build_dataset_sample(dataset_id: int, split_root: Path, chosen_name: str):
    label_dir = find_label_dir(split_root)
    has_labels = label_dir is not None
    dataset = DATALOADER(dataset_id, str(split_root), 
                          ids=[chosen_name], label=has_labels)
    if dataset_id == 2:
        dataset.wsi_level = 3
    return dataset[0], has_labels, label_dir

print('✓ build_dataset_sample() defined')

In [ ]:
# Helper 3: Convert PIL images to tensor batch
def to_tensor_batch(pil_images):
    tensors = [imagenet_transform(img) for img in pil_images]
    return torch.stack(tensors, dim=0).to(DEVICE)

print('✓ to_tensor_batch() defined')

In [ ]:
# Main inference function
def run_single_image_inference(model, image, input_mode, patch_size, 
                               overlap, sub_batch_size, context_m, context_l):
    images = [image]
    local_patches, coordinates, templates, sizes, ratios = global_to_patch(
        images, patch_size, overlap_percentage=overlap
    )

    medium_patches = None
    large_patches = None
    if input_mode in (2, 3):
        medium_patches = global_to_context_patches(images, patch_size, coordinates, context_m)
    if input_mode == 3:
        large_patches = global_to_context_patches(images, patch_size, coordinates, context_l)

    patch_outputs = []
    total_patches = len(local_patches[0])
    print(f'Processing {total_patches} patches...')

    model.eval()
    with torch.no_grad():
        for start in range(0, total_patches, sub_batch_size):
            end = min(start + sub_batch_size, total_patches)
            x_local = to_tensor_batch(local_patches[0][start:end])

            if input_mode == 1:
                out = model(x_local=x_local)
            elif input_mode == 2:
                x_medium = to_tensor_batch(medium_patches[0][start:end])
                out = model(x_local=x_local, x_medium=x_medium)
            elif input_mode == 3:
                x_medium = to_tensor_batch(medium_patches[0][start:end])
                x_large = to_tensor_batch(large_patches[0][start:end])
                out = model(x_local=x_local, x_medium=x_medium, x_large=x_large)
            else:
                raise ValueError(f'Invalid INPUT_MODE: {input_mode}')

            patch_outputs.append(out.detach().cpu().numpy())

    stitched = stitch_patch_predictions_to_global(
        [np.concatenate(patch_outputs, axis=0)],
        N_CLASS,
        sizes,
        coordinates,
        patch_size,
        templates=templates,
    )[0]

    pred_mask = stitched.argmax(axis=0)
    if N_CLASS == 2:
        positive_score = torch.softmax(torch.from_numpy(stitched), dim=0)[1].numpy()
    else:
        positive_score = None

    return {
        'patch_outputs': patch_outputs,
        'stitched_logits': stitched,
        'pred_mask': pred_mask,
        'positive_score': positive_score,
        'sizes': sizes,
        'coordinates': coordinates,
        'templates': templates,
        'ratios': ratios,
    }

print('✓ run_single_image_inference() defined')

In [ ]:
# Load test image
split_root = (DATA_ROOT / SPLIT).resolve()
if not split_root.exists():
    raise FileNotFoundError(f'Split path not found: {split_root}')

chosen_name = find_image_name(split_root, IMAGE_NAME)
sample, has_labels, label_dir = build_dataset_sample(DATASET_ID, split_root, chosen_name)
image = sample['image']
label = sample['label'] if has_labels and 'label' in sample else None

print(f'✓ Image loaded')
print(f'  Name: {chosen_name}')
print(f'  Size: {image.size}')
print(f'  Label dir: {label_dir if label_dir is not None else "not found"}')
if label is not None:
    print(f'  Label size: {label.size}')
else:
    print(f'  No ground-truth mask')

In [ ]:
# Resolve & validate checkpoint path
def resolve_checkpoint_path(ckpt_value):
    if ckpt_value is None:
        return ''
    if isinstance(ckpt_value, Path):
        ckpt_text = str(ckpt_value).strip()
    else:
        ckpt_text = str(ckpt_value).strip()

    if ckpt_text == '':
        return ''

    ckpt_path = Path(ckpt_text)
    if not ckpt_path.is_absolute():
        ckpt_path = (REPO_ROOT / ckpt_path).resolve()

    if not ckpt_path.exists():
        raise FileNotFoundError(f'Checkpoint not found: {ckpt_path}')

    return str(ckpt_path)

resolved_ckpt = resolve_checkpoint_path(CHECKPOINT_PATH)
if REQUIRE_CHECKPOINT and not resolved_ckpt:
    raise ValueError('CHECKPOINT_PATH is empty but REQUIRE_CHECKPOINT=True')

print(f'✓ Checkpoint resolved')
if resolved_ckpt:
    print(f'  Path: {resolved_ckpt}')
else:
    print(f'  Using base pretrained weights')

In [ ]:
# Load model with checkpoint
model = create_model_load_weights(
    n_class=N_CLASS,
    pre_path=resolved_ckpt,
    input_mode=INPUT_MODE,
    use_window=USE_WINDOW,
)

print('✓ Model loaded:', type(model).__name__)

In [ ]:
# Run inference on test image
result = run_single_image_inference(
    model=model,
    image=image,
    input_mode=INPUT_MODE,
    patch_size=PATCH_SIZE,
    overlap=OVERLAP,
    sub_batch_size=SUB_BATCH_SIZE,
    context_m=CONTEXT_M,
    context_l=CONTEXT_L,
)

print('✓ Inference complete')
print(f'  Stitched logits shape: {result["stitched_logits"].shape}')
print(f'  Prediction mask shape: {result["pred_mask"].shape}')

In [ ]:
# Visualize and save results
pred_mask = result['pred_mask']
output_dir = REPO_ROOT / 'notebook'
output_dir.mkdir(exist_ok=True)

# Save the raw prediction for later reuse
pred_mask_path = output_dir / f'{chosen_name}_pred_mask.npy'
np.save(pred_mask_path, pred_mask)

# Save a simple image version of the prediction map
pred_img = Image.fromarray(pred_mask.astype(np.uint8))
pred_png_path = output_dir / f'{chosen_name}_pred_mask.png'
pred_img.save(pred_png_path)

fig, axes = plt.subplots(1, 3 if label is not None else 2, figsize=(14, 5))
if not isinstance(axes, np.ndarray):
    axes = np.array([axes])

# Input image
axes[0].imshow(image)
axes[0].set_title('Input Image')
axes[0].axis('off')

plot_idx = 1

# Ground truth (if available)
if label is not None:
    axes[1].imshow(label.convert('L'), cmap='gray')
    axes[1].set_title('Ground Truth')
    axes[1].axis('off')
    plot_idx = 2

# Predicted mask
axes[plot_idx].imshow(pred_mask, cmap='viridis', vmin=0, vmax=max(1, N_CLASS - 1))
axes[plot_idx].set_title('Predicted Mask')
axes[plot_idx].axis('off')

plt.tight_layout()
plt.show()

print('✓ Saved raw prediction to:', pred_mask_path)
print('✓ Saved PNG prediction to:', pred_png_path)
print('✓ Prediction stored in variable: pred_mask')